# Crypto Cross-Sectional XGBoost ML Factor

**Strategy**: Hourly Binance USDT-M perp futures | XGBoost 5-class classifier | Walk-forward  
**Features**: RSI, ROC, ZscoreRet, ADX, ATR(norm), HistVol, Bollinger %B, VolumeRatio × [7,14,21] + OBV = **25 features**  
**Target**: `forward_return_6` (6-hour forward %) → 5 percentile classes (STRONG_BEAR → STRONG_BULL)  
**Portfolio**: Top-3 longs (STRONG_BULL) + Top-3 shorts (STRONG_BEAR), equal-weight, 6h rebalance  
**Risk**: 5% trailing stop per position | Taker 0.05% / Maker 0.02%

Uses the QTS platform: `DataManager` + `Config.build()` + `FeaturePipeline` + `MLFactorStrategy` + `VectorBTProEngine`.


## Step 0 — Imports

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from datetime import date, datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

from qts.config.builder import Config
from qts.data.sources.binance import BinanceFuturesDataSource
from qts.data.storage.duckdb import DuckDBStorage
from qts.data.manager import DataManager
from qts.orchestration.runtime import build_data_manager
from qts.research.backtest.base import BacktestConfig
from qts.research.backtest.pyfolio_adapter import (
    positions_frame,
    returns_series,
    transactions_frame,
)
from qts.research.backtest.tearsheet import save_tearsheet
from qts.utils.export import export_portfolio_snapshots, export_trade_log
from qts.utils.paths import backtest_exports_dir, database_path, tearsheet_dir

print("Imports OK")


Imports OK


## Step 1 — Fetch & Store Hourly Data

Follows the `store_vn30f1m.ipynb` pattern:  
build `DataManager` directly → `get_futures_ohlcv(interval="1h")` → `upsert_bars()` into DuckDB `crypto_futures_intraday_prices`.

The extended `BinanceFuturesDataSource` now returns `FUTURES_INTRADAY_OHLCV_COLUMNS` (with `bar_time`) for intraday intervals,  
and `DataManager` routes hourly requests to `crypto_futures_intraday_prices` (not `futures_prices`).


In [2]:
import asyncio
import yaml
from qts.data._schemas import FUTURES_INTRADAY_OHLCV_COLUMNS

# --- Configuration ----------------------------------------------------------------
TABLE       = "crypto_futures_intraday_prices"
INTERVAL    = "1h"
FETCH_START = date(2022, 1, 1)
FETCH_END   = date(2026, 5, 31)

with open("../configs/assets/crypto_future.yml") as f:
    SYMBOLS: list[str] = yaml.safe_load(f)["symbols"]
print(f"Universe: {len(SYMBOLS)} symbols")

# --- Build a lightweight DataManager (no Config.build() needed for storage) -------
storage        = DuckDBStorage(database=str(database_path()))
futures_source = BinanceFuturesDataSource.from_env(mode="live")
store_manager  = DataManager(
    stock_source=None,
    crypto_source=None,
    crypto_futures_source=futures_source,
    storage=storage,
)

# --- Fetch directly from BinanceFuturesDataSource ---------------------------------
# This guarantees FUTURES_INTRADAY_OHLCV_COLUMNS schema (bar_time, date, symbol,
# interval, open, high, low, close, volume) and bypasses any stale DuckDB data.
async def _fetch_one(sym: str) -> pl.DataFrame:
    try:
        return await futures_source.get_ohlcv(sym, FETCH_START, FETCH_END, INTERVAL)
    except Exception as exc:
        print(f"  WARN {sym}: {exc}")
        return pl.DataFrame()

frames = await asyncio.gather(*[_fetch_one(sym) for sym in SYMBOLS])
fetched = pl.concat([f for f in frames if f.height > 0], how="vertical").sort(["symbol", "bar_time"])

print(f"Fetched: {fetched.height:,} rows  |  columns: {fetched.columns}")
assert "bar_time" in fetched.columns,  f"bar_time missing — check BinanceFuturesDataSource changes"
assert "interval" in fetched.columns, f"interval missing — check BinanceFuturesDataSource changes"
assert fetched.columns == FUTURES_INTRADAY_OHLCV_COLUMNS, f"Schema mismatch: {fetched.columns}"

# --- Upsert to DuckDB (idempotent: safe to re-run) --------------------------------
store_manager.upsert_bars(
    TABLE,
    fetched,
    sort_by=["bar_time"],
    identity=["symbol", "interval", "bar_time"],
)

fetched.select(
    pl.len().alias("upserted_rows"),
    pl.min("bar_time").alias("first_bar"),
    pl.max("bar_time").alias("last_bar"),
)


Universe: 90 symbols
Fetched: 3,479,850 rows  |  columns: ['bar_time', 'date', 'symbol', 'interval', 'open', 'high', 'low', 'close', 'volume']


upserted_rows,first_bar,last_bar
u32,datetime[μs],datetime[μs]
3479850,2022-01-01 00:00:00,2026-05-31 00:00:00


## Step 2 — Verify Stored Data

In [3]:
where_clause = f"interval = '{INTERVAL}'"
rows = storage.query(f"SELECT * FROM {TABLE} WHERE {where_clause}")
duplicate_key_count = storage.query(f"""
    SELECT symbol, interval, bar_time, COUNT(*) AS row_count
    FROM {TABLE}
    WHERE {where_clause}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").height

symbols_stored = storage.query(f"SELECT DISTINCT symbol FROM {TABLE} WHERE {where_clause} ORDER BY symbol")["symbol"].to_list()

pl.DataFrame({
    "row_count":            [rows.height],
    "symbol_count":         [len(symbols_stored)],
    "is_sorted_by_bar_time":[rows["bar_time"].is_sorted()],
    "duplicate_key_count":  [duplicate_key_count],
    "first_bar":            [str(rows["bar_time"].min())],
    "last_bar":             [str(rows["bar_time"].max())],
})


row_count,symbol_count,is_sorted_by_bar_time,duplicate_key_count,first_bar,last_bar
i64,i64,bool,i64,str,str
3479850,90,true,0,"""2022-01-01 00:00:00""","""2026-05-31 00:00:00"""


## Step 3 — Load YAML Config

`Config.build()` wires: `BinanceFuturesDataSource` → `DuckDBStorage` → extended `FeaturePipeline`  
(new `zscore_ret`, multi-period `adx`/`bollinger pct_b`/`atr_norm`/`volume_ratio`) → `MLFactorStrategy` → `VectorBTProEngine`.


In [4]:
CONFIG_PATH = Path("../configs/strategies/ml_factor/crypto_xgb_crosssectional.yaml")
resolved = Config.build(str(CONFIG_PATH))
config: BacktestConfig = resolved.raw

symbols = config.universe.crypto_futures

print(f"Workflow        : {config.workflow}")
print(f"Universe        : {len(symbols)} crypto futures")
print(f"Date range      : {config.start_date} → {config.end_date}")
print(f"Test start      : {config.test_start_date}")
print(f"Initial capital : {config.initial_capital:,}")
print(f"Engine          : {config.backtest_engine}")
print(f"Train window    : {config.train_window} bars  ({config.train_window // 24} days × 24h)")
print(f"Rebalance freq  : every {config.rebalance_frequency} bars (= {config.rebalance_frequency}h)")
print(f"Benchmark       : {config.benchmark}")


Workflow        : research
Universe        : 66 crypto futures
Date range      : 2022-01-01 → 2025-12-31
Test start      : 2025-10-01
Initial capital : 100,000
Engine          : vectorbt
Train window    : 2880 bars  (120 days × 24h)
Rebalance freq  : every 6 bars (= 6h)
Benchmark       : PERP:BTC/USDT


## Step 4 — Fetch Hourly OHLCV from DuckDB

`data_manager.get_futures_ohlcv(interval="1h")` reads from `crypto_futures_intraday_prices`.  
We then rename `bar_time → date` (keeping `Datetime` precision) so the existing `FeaturePipeline`  
and `MLFactorStrategy` work without modification — they sort by `["symbol", "date"]` which works for Datetime.


In [5]:
data_manager = build_data_manager(resolved)

intraday_raw: pl.DataFrame = await data_manager.get_futures_ohlcv(
    symbols=symbols,
    start=config.start_date,
    end=config.end_date,
    interval="1h",
)

# Drop the day-portion 'date' and 'interval' columns; rename bar_time → date.
# The pipeline sorts by ["symbol", "date"]; Datetime sorts correctly for hourly bars.
raw = (
    intraday_raw
    .drop(["date", "interval"])
    .rename({"bar_time": "date"})
)

print(f"Rows   : {len(raw):,}  |  Symbols: {raw['symbol'].n_unique()}  |  Columns: {raw.columns}")
print(f"date dtype : {raw.schema['date']}  (hourly Datetime — NOT Date)")
raw.head(5)


Rows   : 2,314,224  |  Symbols: 66  |  Columns: ['date', 'symbol', 'open', 'high', 'low', 'close', 'volume']
date dtype : Datetime(time_unit='us', time_zone=None)  (hourly Datetime — NOT Date)


date,symbol,open,high,low,close,volume
datetime[μs],str,f64,f64,f64,f64,f64
2022-01-01 00:00:00,"""PERP:1000SHIB/USDT""",0.033373,0.03389,0.033368,0.033823,5.22172816e8
2022-01-01 00:00:00,"""PERP:1000XEC/USDT""",0.10797,0.11008,0.10731,0.10989,5.874345e6
2022-01-01 00:00:00,"""PERP:1INCH/USDT""",2.3878,2.4206,2.3866,2.4178,700939.0
2022-01-01 00:00:00,"""PERP:AAVE/USDT""",253.95,263.29,253.9,262.22,36441.2
2022-01-01 00:00:00,"""PERP:ADA/USDT""",1.3077,1.3328,1.3076,1.3291,1.461367e7


## Step 5 — Feature Engineering via Platform Pipeline

`resolved.feature_pipeline.fit_transform(raw)` calls the YAML-configured indicator chain:  
`RSI` · `ROC` · `zscore_ret` · `ADX` · `ATR(normalize)` · `hist_vol` · `Bollinger(pct_b)` · `volume_ratio` · `OBV` · `ForwardReturns([6])`.


In [ ]:
feature_pipeline = resolved.feature_pipeline
featured: pl.DataFrame = feature_pipeline.fit_transform(raw)

predictor_cols = config.strategy.params["predictor_cols"]
target_col     = config.strategy.params["target_col"]

print(f"Featured shape  : {featured.shape}")
print(f"Feature columns : {predictor_cols}")
print(f"Target column   : {target_col}")

null_rates = {
    col: featured[col].null_count() / len(featured)
    for col in predictor_cols + [target_col]
}
print(f"\nNull rates (features): max = {max(null_rates[c] for c in predictor_cols):.1%}")
print(f"Null rate  (target) : {null_rates[target_col]:.1%}")

featured.select(["date", "symbol"] + predictor_cols[:3] + [target_col]).head(3)


Featured shape  : (2314224, 33)
Feature columns : ['rsi_7', 'rsi_14', 'rsi_21', 'roc_7', 'roc_14', 'roc_21', 'zscore_ret_7', 'zscore_ret_14', 'zscore_ret_21', 'adx_7', 'adx_14', 'adx_21', 'atr_norm_7', 'atr_norm_14', 'atr_norm_21', 'hist_vol_7', 'hist_vol_14', 'hist_vol_21', 'bb_pct_b_7', 'bb_pct_b_14', 'bb_pct_b_21', 'vol_ratio_7', 'vol_ratio_14', 'vol_ratio_21', 'obv']
Target column   : forward_return_6

Null rates (features): max = 0.1%
Null rate  (target) : 0.0%


date,symbol,rsi_7,rsi_14,rsi_21,forward_return_6
datetime[μs],str,f64,f64,f64,f64
2022-01-01 00:00:00,"""PERP:1000SHIB/USDT""",null,null,null,-0.004346
2022-01-01 01:00:00,"""PERP:1000SHIB/USDT""",null,null,null,-0.000473
2022-01-01 02:00:00,"""PERP:1000SHIB/USDT""",null,null,null,-0.001897


## Step 6 — Feature Exploration

In [7]:
import numpy as np

feat_pd = featured.select(predictor_cols).to_pandas()
corr = feat_pd.corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(predictor_cols)))
ax.set_yticks(range(len(predictor_cols)))
ax.set_xticklabels(predictor_cols, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(predictor_cols, fontsize=7)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Feature correlation matrix (25 × 25)")
plt.tight_layout()
plt.show()

featured.select(predictor_cols).describe()


statistic,rsi_7,rsi_14,rsi_21,roc_7,roc_14,roc_21,zscore_ret_7,zscore_ret_14,zscore_ret_21,adx_7,adx_14,adx_21,atr_norm_7,atr_norm_14,atr_norm_21,hist_vol_7,hist_vol_14,hist_vol_21,bb_pct_b_7,bb_pct_b_14,bb_pct_b_21,vol_ratio_7,vol_ratio_14,vol_ratio_21,obv
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.313828e6,2.313366e6,2.312904e6,2.313762e6,2.3133e6,2.312838e6,2.313762e6,2.3133e6,2.312838e6,2.313432e6,2.312508e6,2.311584e6,2.313828e6,2.313366e6,2.312904e6,2.313762e6,2.3133e6,2.312838e6,2.313828e6,2.313366e6,2.312904e6,2.313828e6,2.313366e6,2.312904e6,2.314224e6
"""null_count""",396.0,858.0,1320.0,462.0,924.0,1386.0,462.0,924.0,1386.0,792.0,1716.0,2640.0,396.0,858.0,1320.0,462.0,924.0,1386.0,396.0,858.0,1320.0,396.0,858.0,1320.0,0.0
"""mean""",49.801893,49.676078,49.62169,-0.000034,-0.000075,-0.000113,-0.00701,-0.00453,-0.004302,35.916389,26.84231,22.754359,0.014432,0.014444,0.014457,0.007969,0.008621,0.008882,0.50054,0.498582,0.496641,1.017071,1.024972,1.025204,5.7468e9
"""std""",16.209711,11.684116,9.647042,0.028015,0.039473,0.048465,1.001371,1.006617,1.010429,13.948054,11.269177,9.848502,0.008204,0.00757,0.00725,0.005988,0.005752,0.00558,0.294463,0.304568,0.308674,0.546574,0.696795,0.781434,2.7514e10
"""min""",1.209474,5.028751,7.527766,-0.505152,-0.554379,-0.569731,-2.449482,-3.603419,-4.46348,6.483616,4.666094,3.617841,0.000664,0.000833,0.001064,0.0,0.000359,0.000429,0.0,0.0,0.0,0.01633,0.017236,0.017763,-1.1827e11
"""25%""",38.39718,41.838262,43.252975,-0.012148,-0.018235,-0.023469,-0.730438,-0.641304,-0.607318,25.283099,18.350027,15.389971,0.009326,0.009644,0.009816,0.004329,0.005071,0.005422,0.247498,0.239794,0.233936,0.659621,0.604203,0.581139,-1.3552e8
"""50%""",49.834002,49.686339,49.586206,0.0,-0.000242,-0.000596,1.3684e-15,0.007348,0.008632,33.367823,24.499362,20.623115,0.012688,0.0129,0.013005,0.006498,0.007286,0.007633,0.501664,0.497759,0.494837,0.890145,0.845838,0.818658,3.6478e6
"""75%""",61.17472,57.465072,55.947071,0.011908,0.017282,0.021538,0.720804,0.64518,0.615382,44.258292,33.132619,28.069826,0.017366,0.017309,0.017298,0.009817,0.010547,0.010813,0.75485,0.758573,0.76008,1.220869,1.218476,1.204149,1.0336e9
"""max""",99.655627,98.880407,98.026298,1.07346,1.052168,1.90483,2.449473,3.603714,4.467085,98.479639,85.481327,75.834349,0.361249,0.241435,0.208454,0.235018,0.177651,0.146959,1.0,1.0,1.0,6.958335,13.802747,20.576794,2.5762e11


## Step 7 — Inspect Resolved MLFactorStrategy

In [ ]:
strategy = resolved.strategy
model    = getattr(strategy, "model", None)

print(f"Strategy         : {strategy.__class__.__name__}")
print(f"Model            : {model.__class__.__name__ if model else 'train_func'}")
print(f"Predictor cols   : {strategy.predictor_cols}")
print(f"Target col       : {strategy.target_col}")
print(f"Rebalance period : {strategy.rebalance_period} bars  (= {strategy.rebalance_period}h)")
print(f"cv_splits        : {strategy.cv_splits}")
print(f"cv_max_train_size: {strategy.cv_max_train_size}  ({strategy.cv_max_train_size // 24 if strategy.cv_max_train_size else '—'} days)")


Strategy         : MLFactorStrategy
Model            : XGBClassifierModel
Predictor cols   : ['rsi_7', 'rsi_14', 'rsi_21', 'roc_7', 'roc_14', 'roc_21', 'zscore_ret_7', 'zscore_ret_14', 'zscore_ret_21', 'adx_7', 'adx_14', 'adx_21', 'atr_norm_7', 'atr_norm_14', 'atr_norm_21', 'hist_vol_7', 'hist_vol_14', 'hist_vol_21', 'bb_pct_b_7', 'bb_pct_b_14', 'bb_pct_b_21', 'vol_ratio_7', 'vol_ratio_14', 'vol_ratio_21', 'obv']
Target col       : forward_return_6
Rebalance period : 6 bars  (= 6h)
cv_splits        : 3
cv_max_train_size: 2880  (120 days)


: 

## Step 8 — Run Backtest via VectorBTProEngine

`engine.run(pipeline=..., ohlcv=raw)` triggers `walk_forward_signals`:  
- Iterates over all `Datetime` rebalance points (every 6h)  
- For each: slice rolling 2880-bar window → `feature_pipeline.fit_transform` → `MLFactorStrategy.generate_signals`  
  → top-3 longs + top-3 shorts by XGBoost class score  
- VectorBTProEngine runs `Portfolio.from_orders` with the collected signal schedule


In [ ]:
engine = resolved.engine
print(f"Engine: {engine.__class__.__name__}")

result = engine.run(
    strategy=strategy,
    data=featured,
    config=config,
    pipeline=feature_pipeline,
    ohlcv=raw,
)

print("\n=== Overall metrics ===")
for key, value in result.metrics.items():
    print(f"  {key:20s}: {value:.4f}")

if result.metrics_is:
    print("\n=== In-sample metrics ===")
    for key, value in result.metrics_is.items():
        print(f"  {key:20s}: {value:.4f}")

if result.metrics_oos:
    print("\n=== Out-of-sample metrics ===")
    for key, value in result.metrics_oos.items():
        print(f"  {key:20s}: {value:.4f}") 


Engine: VectorBTProEngine


## Step 9 — IS vs OOS Equity Curve

In [1]:
from qts.research.backtest.pyfolio_adapter import returns_series

oos_start = pd.Timestamp(config.test_start_date).tz_localize("UTC") if config.test_start_date else None
rets = returns_series(result)

is_rets  = rets[rets.index < oos_start]  if oos_start is not None else rets
oos_rets = rets[rets.index >= oos_start] if oos_start is not None else pd.Series(dtype=float)

def annualised_sharpe(s: pd.Series, bars_per_year: float = 24 * 365) -> float:
    if s.empty or s.std() == 0:
        return 0.0
    return (s.mean() / s.std()) * (bars_per_year ** 0.5)

def period_label(s: pd.Series) -> str:
    return f"{s.index[0].date()} → {s.index[-1].date()}" if not s.empty else "empty"

print(f"IS  period : {period_label(is_rets)}   |  Sharpe: {annualised_sharpe(is_rets):.3f}")
print(f"OOS period : {period_label(oos_rets)}  |  Sharpe: {annualised_sharpe(oos_rets):.3f}")

equity = result.equity_curve.to_pandas()
equity["date"] = pd.to_datetime(equity["date"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(equity["date"], equity["equity"] / equity["equity"].iloc[0],
        label="Strategy", linewidth=1.5)
if oos_start is not None:
    ax.axvline(oos_start, color="red", linestyle="--", alpha=0.7, label="OOS start")
ax.set_title("Equity curve (hourly → engine-aggregated)")
ax.set_ylabel("Growth of initial capital")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
   
   

NameError: name 'config' is not defined

## Step 10 — Save CSV Outputs

In [ ]:
from datetime import datetime as _dt

run_id = f"{result.engine_name}_{_dt.utcnow().strftime('%Y%m%dT%H%M%SZ')}"
exports_dir = backtest_exports_dir()

tl_path  = exports_dir / f"{run_id}_trade_log.csv"
snp_path = exports_dir / f"{run_id}_snapshots.csv"

export_trade_log(result, tl_path)
export_portfolio_snapshots(result, snp_path)

print(f"Trade log  → {tl_path}  ({result.trade_log.shape[0]} rows)")
print(f"Snapshots  → {snp_path}  ({result.portfolio_snapshots.shape[0]} rows)")
result.trade_log.head(5)
    
     

## Step 11 — Generate Pyfolio Tearsheet

In [ ]:
benchmark_rets = None
if config.benchmark:
    from qts.orchestration.flow import _fetch_benchmark_returns  # noqa: PLC0415

    benchmark_rets = _fetch_benchmark_returns(
        config.benchmark,
        config.start_date,
        config.end_date,
        data_manager,
    )
    if benchmark_rets is not None:
        print(f"Benchmark loaded: {len(benchmark_rets)} bars of {config.benchmark}")
    else:
        print(f"Benchmark {config.benchmark} not found in DB — running without")

pdf_path = save_tearsheet(
    result=result,
    out_dir=tearsheet_dir(),
    run_id=run_id,
    benchmark_rets=benchmark_rets,
)

if pdf_path:
    print(f"Tearsheet PDF → {pdf_path}  ({pdf_path.stat().st_size / 1024:.1f} KB)")
else:
    print("Tearsheet skipped (pyfolio not installed)")
    

    